## 1. 安装


In [ ]:
%cd /data/projects_ING/freqtrade


In [ ]:
!uv venv 


## 2. 初始化项目


In [ ]:
%%bash
# 创建用户目录
freqtrade create-userdir --userdir user_data

# 创建配置文件
freqtrade new-config --config user_data/config.json

# 创建 Aberration 策略模板
freqtrade new-strategy --strategy Aberration --userdir user_data


## 4. 策略开发


In [ ]:
from freqtrade.strategy import IStrategy
from pandas import DataFrame
import talib.abstract as ta

class SimpleStrategy(IStrategy):
    timeframe = '5m'
    stoploss = -0.10
    
    def populate_indicators(self, dataframe: DataFrame, metadata: dict) -> DataFrame:
        dataframe['rsi'] = ta.RSI(dataframe, timeperiod=14)
        return dataframe
    
    def populate_entry_trend(self, dataframe: DataFrame, metadata: dict) -> DataFrame:
        dataframe.loc[(dataframe['rsi'] < 30), 'enter_long'] = 1
        return dataframe
    
    def populate_exit_trend(self, dataframe: DataFrame, metadata: dict) -> DataFrame:
        dataframe.loc[(dataframe['rsi'] > 70), 'exit_long'] = 1
        return dataframe


## 5. 下载数据


In [ ]:
!uv run freqtrade download-data \
--exchange binance \
--pairs BTC/USDT:USDT \
--timeframes 1m \
--days 7000


## 6. 回测


In [ ]:
%%bash
freqtrade backtesting \
  --strategy Aberration


## 7. 参数优化


In [ ]:
%%bash
# Hyperopt 优化
freqtrade hyperopt \
  --strategy Aberration \
  --hyperopt-loss SharpeHyperOptLoss \
  --epochs 100 \
  --spaces buy


## 8. 实盘交易

```bash
# Dry Run (模拟)
freqtrade trade --config config.json --strategy SimpleStrategy

# Live Trading (实盘)
# 修改 config.json: "dry_run": false
freqtrade trade --config config.json --strategy SimpleStrategy
```


## 9. 分析结果


In [ ]:
from freqtrade.data.btanalysis import load_backtest_data, load_backtest_stats
import pandas as pd

# 加载回测结果
stats = load_backtest_stats('user_data/backtest_results/backtest-result-2026-01-29_21-03-41.zip')
trades = load_backtest_data('user_data/backtest_results/backtest-result-2026-01-29_21-03-41.zip')

# 查看统计
print(f"总收益: {stats['strategy']['SimpleStrategy']['profit_total']:.2%}")
print(f"胜率: {stats['strategy']['SimpleStrategy']['wins']/stats['strategy']['SimpleStrategy']['total_trades']:.2%}")
print(f"最大回撤: {stats['strategy']['SimpleStrategy']['max_drawdown']:.2%}")


In [ ]:
!freqtrade trade  --strategy Aberration


## 10. 绘图分析

```bash
# 生成图表
freqtrade plot-dataframe \
  --config config.json \
  --strategy SimpleStrategy \
  --pairs BTC/USDT \
  --indicators1 rsi \
  --timerange 20230101-20230131

# 生成收益曲线
freqtrade plot-profit \
  --strategy Aberration \
  --timeframe 4h
```


## 11. 常用命令

```bash
# 查看策略列表
freqtrade list-strategies

# 测试策略
freqtrade test-pairlist --config config.json

# 查看交易对
freqtrade list-pairs --exchange binance

# 查看市场数据
freqtrade list-markets --exchange binance
```


## 12. 高级功能

### 自定义止损

```python
def custom_stoploss(self, pair: str, trade: Trade, current_time: datetime,
                    current_rate: float, current_profit: float, **kwargs) -> float:
    if current_profit > 0.10:
        return -0.05  # 盈利10%后，止损设为-5%
    return -0.10  # 默认止损-10%
```

### 自定义仓位

```python
def custom_stake_amount(self, pair: str, current_time: datetime, current_rate: float,
                        proposed_stake: float, min_stake: float, max_stake: float,
                        **kwargs) -> float:
    dataframe, _ = self.dp.get_analyzed_dataframe(pair, self.timeframe)
    if dataframe['rsi'].iloc[-1] < 20:
        return proposed_stake * 2  # RSI极低时加倍仓位
    return proposed_stake
```


## 13. 监控与通知

```json
{
  "telegram": {
    "enabled": true,
    "token": "your_bot_token",
    "chat_id": "your_chat_id"
  },
  "webhook": {
    "enabled": true,
    "url": "https://your-webhook-url",
    "webhookbuy": { "value1": "Buy {pair}" },
    "webhooksell": { "value1": "Sell {pair}" }
  }
}
```
